# Playground-Testbereich einmalig anlegen

Dieses Notebook richtet ausschließlich den **synthetischen FB-8.1-Testbereich** ein.
**Run All zeigt eine lokale Vorschau.** Die letzte Zelle bleibt deaktiviert.

Geplant sind zwei Projekte `GASI_TEST` mit je einer Collection:
- Laborbuch: im eigenen `8.1_...`-Space, Collection `RUNS`.
- Inventar: in `8.1_EQUIPMENT`, Collection `INSTRUMENTS`.

Für das fiktive Instrument wird ein bestehender gültiger Standortcode ausschließlich
als **technischer Test-Platzhalter** verwendet. Er behauptet keinen realen Gerätestandort.
Es werden keine globalen Wertelisten verändert.
Vorhandene Stammdaten, Zugriffsrechte und andere Bereiche werden nicht verändert.
Dieses Notebook legt noch keinen Messlauf und kein Dataset an; dafür ist Notebook 05 zuständig.

In [ ]:
import gasi_openbis_playground as playground
import gasi_playground_provision as provision

current_config, message = playground.load_target_config()
print(message)

# Bei einer neuen Arbeitskopie hier den eigenen FB-8.1-Laborbuch-Space
# und den ausdrücklichen Auftrag zur Einrichtung eintragen.
LAB_SPACE = (current_config or {}).get("lab_notebook", {}).get("space", "")
APPROVAL_REFERENCE = (current_config or {}).get("approved_test_scope", {}).get("approval_reference", "")

location = (current_config or {}).get("synthetic_location", {})
LOCATION_CODE = location.get("code", "")
LOCATION_LABEL = location.get("label", "")
plan = provision.proposed_config(LAB_SPACE, APPROVAL_REFERENCE, LOCATION_CODE, LOCATION_LABEL) if LAB_SPACE else None
if plan is None:
    print("LAB_SPACE aus Notebook 04 übernehmen; Freigabereferenz in APPROVAL_REFERENCE dokumentieren.")
else:
    playground.show_plan(plan)
    print("Zusätzlich: je ein GASI_TEST-Projekt, zwei Collections und ein ausdrücklich gekennzeichneter bestehender Standort-Testwert.")

## Einrichtung bewusst starten

Nach Prüfung der Vorschau `PROVISION_ENABLED = True` setzen und die Phrase
`CREATE GASI FB81 PLAYGROUND TEST AREA` eintragen. Dann nur die nächste Zelle ausführen.
Sie prüft Host, exakte FB-8.1-Ziele, Testmarker, Typen, Pflichtfelder und Wertelisten.
Bestehende Testbereiche werden wiederverwendet. Nach einem Teilfehler dieselbe
Konfiguration erneut ausführen. Fremde Einträge werden nicht überschrieben.

Erst nach erfolgreichem Zurücklesen wird die gemeinsame
`config/playground_test_target.json` gespeichert. Anschließend Notebook 04 und 05
mit frischem Kernel ausführen. Der Auftrag ist eine auf diese Ziele begrenzte
Testfreigabe; tatsächliche Serverrechte werden weder hinzugefügt noch vorgetäuscht.

In [ ]:
PROVISION_ENABLED = False
PROVISION_CONFIRMATION = ""

if not PROVISION_ENABLED:
    print("Einrichtung deaktiviert. Nur lokale Vorschau; kein Playground-Schreibzugriff.")
elif plan is None:
    print("Einrichtung gesperrt: Laborbuch-Space fehlt.")
else:
    import json
    result, message = provision.provision_playground(plan, PROVISION_CONFIRMATION)
    print(message)
    if result is not None:
        print(json.dumps({k: v for k, v in result.items() if k != "config"}, ensure_ascii=False, indent=2))